In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)

# =====================================================
# LOAD DATA
# =====================================================

train = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

# Fill missing values
for col in ["prompt", "A", "B", "C", "D", "E"]:
    train[col] = train[col].fillna("")

# =====================================================
# LABELS
# =====================================================

label2id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

train["labels"] = train["answer"].map(label2id)

# =====================================================
# Split data
# =====================================================

train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

In [ ]:
# =====================================================
# Model
# =====================================================

MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# =====================================================
# Tokenization
# =====================================================

OPTIONS = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):

    first_sentences = []
    second_sentences = []

    for i in range(len(examples["prompt"])):

        prompt = examples["prompt"][i]

        choices = [
            examples["A"][i],
            examples["B"][i],
            examples["C"][i],
            examples["D"][i],
            examples["E"][i]
        ]

        first_sentences.extend([prompt] * 5)
        second_sentences.extend(choices)

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length"
    )

    result = {}

    for k, v in tokenized.items():
        result[k] = [v[i:i + 5] for i in range(0, len(v), 5)]

    return result


In [ ]:
# =====================================================
# Apply Tokenization
# =====================================================

train_ds = Dataset.from_pandas(
    train_df.reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df.reset_index(drop=True)
)

train_ds = train_ds.map(
    preprocess_function,
    batched=True
)

val_ds = val_ds.map(
    preprocess_function,
    batched=True
)

columns = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_ds.set_format(
    type="torch",
    columns=columns
)

val_ds.set_format(
    type="torch",
    columns=columns
)

print(train_ds[0])


In [ ]:
# =====================================================
# Model Load
# =====================================================

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

In [ ]:
# =====================================================
# Metrics calculation
# =====================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    macro_f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

In [ ]:
# =====================================================
# Training Parameters
# =====================================================

args = TrainingArguments(
    output_dir="./deberta",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    weight_decay=0.01,

    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
)

In [ ]:
# =====================================================
# Trainer
# =====================================================

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)


In [ ]:
batch = train_ds[:2]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device)
    )

print("NaN logits:",
      torch.isnan(outputs.logits).any())

print(outputs.logits[:2])

In [ ]:
# =====================================================
# VALIDATION
# =====================================================

trainer.train()
metrics = trainer.evaluate()

print(metrics)


In [ ]:
# Prediction for Submission

test = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

for col in ["prompt", "A", "B", "C", "D", "E"]:
    test[col] = test[col].fillna("")

test_ds = Dataset.from_pandas(
    test.reset_index(drop=True)
)

test_ds = test_ds.map(
    preprocess_function,
    batched=True
)

test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

predictions = trainer.predict(test_ds)

pred_ids = np.argmax(
    predictions.predictions,
    axis=1
)

pred_labels = [
    id2label[p]
    for p in pred_ids
]


In [ ]:
# Submission

submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

submission["Prediction"] = pred_labels

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("submission.csv saved")
print(submission.head())